# Module 5 (Advanced): RAG Evaluation & Failure Analysis

## Objective

We evaluate RAG systems across:

1. Retrieval Quality
2. Faithfulness (Grounding)
3. Answer Quality


In [23]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters langsmith

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

## Setup Azure OpenAI environment variables & LLM and embedding models 
Copy content from previous module

In [25]:
# Initialize environment variables for Azure OpenAI
api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()


# Create embeddings client based on endpoint type
# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized OpenAI-compatible endpoint.")

    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")

Embeddings client initialized OpenAI-compatible endpoint.
LLM initialized via OpenAI-compatible endpoint.


In [26]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain helps build RAG systems using LLMs and retrievers.",
        metadata={"source": "langchain.pdf", "page": 1}
    ),
    Document(
        page_content="LangGraph enables multi-step workflows and agent orchestration.",
        metadata={"source": "langgraph.pdf", "page": 2}
    ),
    Document(
        page_content="LangSmith provides observability and evaluation for LLM systems.",
        metadata={"source": "langsmith.pdf", "page": 3}
    ),
]

In [27]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [28]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

In [29]:
from langchain_chroma import Chroma
# Embeddings and Persistent DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [30]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})\n{doc.page_content}"
        for doc in docs
    )

In [31]:
def hybrid_retrieve(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = retriever.invoke(query)
    
    # Combine and deduplicate
    combined = bm25_docs + vector_docs
    
    unique_docs = {doc.page_content: doc for doc in combined}
    
    return list(unique_docs.values())

In [32]:
def simple_reranker(query, docs):
    def score(doc):
        return sum(word in doc.page_content.lower() for word in query.lower().split())
    
    ranked = sorted(docs, key=score, reverse=True)
    return ranked[:3]

## Problem Example

User Query:
"What does it do?"

This query is too vague for retrieval.

We need to transform it into something like:
"What does LangGraph do in LLM systems?"

In [33]:
def rewrite_query(query):
    rewrite_prompt = ChatPromptTemplate.from_template("""
    Rewrite the following query to be more specific and retrieval-friendly.

    Original Query:
    {query}

    Improved Query:
    """)
    
    response = llm.invoke(
        rewrite_prompt.invoke({"query": query})
    )
    
    return response.content.strip()

In [34]:
query = "What does it do?"
improved_query = rewrite_query(query)

print("Original:", query)
print("Rewritten:", improved_query)

Original: What does it do?
Rewritten: Improved Query:
What is the function and purpose of **[the specific feature/tool/system]**, and what are the key tasks it performs and outputs it produces?


## Multi-Query Generation

Instead of one query, generate multiple variations.

In [35]:
def generate_multi_queries(query):
    multi_prompt = ChatPromptTemplate.from_template("""
    Generate 3 different variations of the query for better document retrieval.

    Query:
    {query}
    
    Variations:
    """)
    
    response = llm.invoke(
        multi_prompt.invoke({"query": query})
    )
    
    queries = response.content.split("\n")
    return [q.strip("- ").strip() for q in queries if q.strip()]

## Combine Multi-Query Retrieval

In [36]:
def multi_query_retrieve(query):
    queries = generate_multi_queries(query)
    
    all_docs = []
    for q in queries:
        docs = hybrid_retrieve(q)
        all_docs.extend(docs)
    
    # Deduplicate
    unique_docs = {doc.page_content: doc for doc in all_docs}
    
    return list(unique_docs.values())

## Rerank the Combined Results

In [37]:
def full_retrieval_pipeline(query):
    rewritten = rewrite_query(query)
    docs = multi_query_retrieve(rewritten)
    reranked = simple_reranker(rewritten, docs)
    
    return reranked

In [38]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Include source citations in your answer.

Context:
{context}

Question:
{question}
""")

In [39]:
def ask_advanced_rag(query):
    docs = full_retrieval_pipeline(query)
    
    context = format_docs(docs)
    
    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    
    return {
        "answer": response.content,
        "docs": docs
    }

## Step 1: Create Evaluation Dataset

Each test case includes:
- question
- expected answer
- expected keywords

In [40]:
eval_dataset = [
    {
        "question": "What does LangGraph do?",
        "ground_truth": "LangGraph is used for orchestrating workflows and agents",
    },
    {
        "question": "What is LangSmith used for?",
        "ground_truth": "LangSmith helps with debugging, evaluation, and observability of LLM applications",
    }
]

## Step 2: Evaluation Prompt (LLM-as-a-Judge)

This is what RAGAS uses internally.

In [41]:
def judge_answer(question, answer, ground_truth):
    eval_prompt = f"""
    You are an evaluator.

    Question: {question}
    Ground Truth: {ground_truth}
    Model Answer: {answer}

    Score the answer from 0 to 1 based on correctness and completeness.

    Only return a number.
    """
    
    response = llm.invoke(eval_prompt)
    return float(response.content.strip())

## Step 3: Check Faithfullness

In [42]:
# Ground Checker for Faithfulness
def check_faithfulness(answer, context):
    prompt = f"""
    Check if the answer is supported by the context.

    Context:
    {context}

    Answer:
    {answer}

    Return YES or NO.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

## Step 4: Retrieval Quality Check

In [43]:
def check_retrieval(question, docs):
    context = "\n\n".join([d.page_content for d in docs])
    
    prompt = f"""
    Question: {question}

    Context:
    {context}

    Does the context contain enough information to answer the question?

    Answer YES or NO.
    """
    
    response = llm.invoke(prompt)
    return response.content.strip()

## Step 5: Full Evaluation Pipeline

In [44]:
def evaluate_rag_system(dataset):
    results = []
    
    for item in dataset:
        rag_result = ask_advanced_rag(item["question"])
        
        answer = rag_result["answer"]
        docs = rag_result["docs"]
        context = format_docs(docs)
        
        correctness = judge_answer(
            item["question"], answer, item["ground_truth"]
        )
        
        faithfulness = check_faithfulness(answer, context)
        retrieval = check_retrieval(item["question"], docs)
        
        results.append({
            "question": item["question"],
            "correctness": correctness,
            "faithfulness": faithfulness,
            "retrieval": retrieval
        })
    
    return results

In [45]:
def classify_failure(result):
    if result["retrieval"] == "NO":
        return "Retrieval Failure"
    elif result["faithfulness"] == "NO":
        return "Hallucination / Grounding Failure"
    elif result["correctness"] < 0.5:
        return "Generation Failure"
    else:
        return "Pass"

In [46]:
results = evaluate_rag_system(eval_dataset)

for r in results:
    failure_type = classify_failure(r)
    
    print("\nQuestion:", r["question"])
    print("Correctness:", r["correctness"])
    print("Faithfulness:", r["faithfulness"])
    print("Retrieval:", r["retrieval"])
    print("Failure Type:", failure_type)


Question: What does LangGraph do?
Correctness: 1.0
Faithfulness: YES
Retrieval: YES
Failure Type: Pass

Question: What is LangSmith used for?
Correctness: 0.9
Faithfulness: YES
Retrieval: YES
Failure Type: Pass
